## Import

In [21]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

import optuna
from sklearn.model_selection import GridSearchCV 

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, max_error

import pandas as pd

In [12]:
from ML.utils.utils import *
from ML.utils.Data_preparator import Data_preparator
from ML.utils.Model_evaluator import Model_evaluator
from ML.utils.Model_trainer import Model_trainer
from physics.Iso_data_handler import Iso_data_handler
from physics.Data_visualiser import Data_visualiser

In [13]:
pre_path = "../../../../../../../"
physical_model = "MIST"
path_to_data = pre_path + "data/MIST_v1.2_vvcrit0.0_basic_isos/"
path_to_results = pre_path + "results/model_A/fine_tuned_models/"
path_to_predictions = pre_path + "predictions/model_A/fine_tuned_models/"
tag = "Base"
output_parameters = ["mass", "radius"]

## Data preparation

In [14]:
iso_handler = Iso_data_handler(path_to_data, 
                              ['log10_isochrone_age_yr', 'log_Teff', 'log_g', 'phase', 'metallicity', 'star_mass', 'log_R'], 
                              physical_model, reclassify=True)

iso_df = iso_handler.get_isochrone_dataframe()

Reading MIST dataframe from csv file...


In [15]:
phase_filtered_iso_df = Data_preparator.filter_data(iso_df, {'phase':[0, 2, 3, 4, 5]})

In [16]:
X_train, X_test, y_train, y_test = \
    Data_preparator.split_data(phase_filtered_iso_df, x_cols=['log10_isochrone_age_yr', 'log_Teff', 'log_g', 'metallicity'], 
                               y_cols=['star_mass', 'log_R'], random_state=12, print_stats=True)

Training set statistics:
Range in train data for the star_mass parameter : 0.0999979840073621 - 298.5447575808816
Median value in train data for the star_mass parameter: 2.0816081316727946
Mean value in train data for the star_mass parameter: 7.558407372495925

Range in train data for the log_R parameter : -0.9974747647513328 - 3.129269620812593
Median value in train data for the log_R parameter: 1.4993114860984695
Mean value in train data for the log_R parameter: 1.3944707591667809

Testing set statistics:
Range in test data for the star_mass parameter : 0.0999981896729906 - 296.5221171165397
Median value in test data for the star_mass parameter: 2.082595606409119
Mean value in test data for the star_mass parameter: 7.471864103970097

Range in test data for the log_R parameter : -0.9974234436680278 - 3.1297545143214007
Median value in test data for the log_R parameter: 1.5026448988619927
Mean value in test data for the log_R parameter: 1.396340263115711



## Fine-tuning

### XGBoost

#### Tuning the booster and n_estimators parameters

In [28]:
scoring = ["neg_root_mean_squared_error", "neg_mean_absolute_error"] # RMSE and MAE

cv_params = {'booster': ["gbtree", "dart"],
             'n_estimators': [100, 150, 200, 250, 300]
            }

gscv_xgb_1 = GridSearchCV(XGBRegressor(),
                          param_grid = cv_params,
                          scoring = scoring,
                          cv = 10,
                          verbose = 2,
                          n_jobs = 10,
                          refit="neg_root_mean_squared_error"
)

gscv_xgb_1.fit(X_train,y_train)
print(gscv_xgb_1.cv_results_)
df = pd.DataFrame.from_dict(gscv_xgb_1.cv_results_)
display(df)
print(f"Best parameters {gscv_xgb_1.best_params_}")
print(f"Best score {gscv_xgb_1.best_score_}" )


Fitting 10 folds for each of 10 candidates, totalling 100 fits
{'mean_fit_time': array([  20.69389312,   29.18247805,   38.35277479,   47.2308882 ,
         60.62191837,  356.08439801,  895.44667237, 1507.51369855,
       2402.25628304, 3366.69091668]), 'std_fit_time': array([0.07769418, 0.06070511, 0.13790512, 0.22198645, 0.30830806,
       0.85301046, 0.74887913, 1.29224332, 1.59952764, 1.82418495]), 'mean_score_time': array([0.26689737, 0.39263556, 0.5014365 , 0.6506825 , 0.76257923,
       1.46083403, 2.28069382, 2.9225358 , 3.89363177, 4.24273353]), 'std_score_time': array([0.0108491 , 0.01546095, 0.03380829, 0.0486309 , 0.03812669,
       0.01652387, 0.05503893, 0.04087245, 0.02967931, 0.33853908]), 'param_booster': masked_array(data=['gbtree', 'gbtree', 'gbtree', 'gbtree', 'gbtree',
                   'dart', 'dart', 'dart', 'dart', 'dart'],
             mask=[False, False, False, False, False, False, False, False,
                   False, False],
       fill_value=np.str_('?')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_booster,param_n_estimators,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,...,split3_test_neg_mean_absolute_error,split4_test_neg_mean_absolute_error,split5_test_neg_mean_absolute_error,split6_test_neg_mean_absolute_error,split7_test_neg_mean_absolute_error,split8_test_neg_mean_absolute_error,split9_test_neg_mean_absolute_error,mean_test_neg_mean_absolute_error,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error
0,20.693893,0.077694,0.266897,0.010849,gbtree,100,"{'booster': 'gbtree', 'n_estimators': 100}",-0.862622,-0.815595,-0.833992,...,-0.210185,-0.206682,-0.205544,-0.212830,-0.211501,-0.215420,-0.205422,-0.208992,0.003579,9
1,29.182478,0.060705,0.392636,0.015461,gbtree,150,"{'booster': 'gbtree', 'n_estimators': 150}",-0.771157,-0.735993,-0.758892,...,-0.192054,-0.189733,-0.189802,-0.193274,-0.190086,-0.196253,-0.189847,-0.191225,0.002417,7
2,38.352775,0.137905,0.501436,0.033808,gbtree,200,"{'booster': 'gbtree', 'n_estimators': 200}",-0.721268,-0.696496,-0.721882,...,-0.178115,-0.173815,-0.177536,-0.180543,-0.177889,-0.179410,-0.177139,-0.177351,0.002094,5
3,47.230888,0.221986,0.650682,0.048631,gbtree,250,"{'booster': 'gbtree', 'n_estimators': 250}",-0.693030,-0.667542,-0.696751,...,-0.168372,-0.164750,-0.167782,-0.169838,-0.166401,-0.170032,-0.168284,-0.167540,0.002249,4
4,60.621918,0.308308,0.762579,0.038127,gbtree,300,"{'booster': 'gbtree', 'n_estimators': 300}",-0.676538,-0.652146,-0.683637,...,-0.161682,-0.156509,-0.158714,-0.162931,-0.157452,-0.158893,-0.159840,-0.159277,0.002054,1
5,356.084398,0.853010,1.460834,0.016524,dart,100,"{'booster': 'dart', 'n_estimators': 100}",-0.862622,-0.815595,-0.833992,...,-0.210185,-0.206682,-0.205544,-0.212830,-0.211501,-0.215420,-0.205422,-0.208992,0.003579,10
6,895.446672,0.748879,2.280694,0.055039,dart,150,"{'booster': 'dart', 'n_estimators': 150}",-0.771157,-0.735993,-0.758892,...,-0.192054,-0.189733,-0.189802,-0.193274,-0.190086,-0.196253,-0.189847,-0.191225,0.002417,8
7,1507.513699,1.292243,2.922536,0.040872,dart,200,"{'booster': 'dart', 'n_estimators': 200}",-0.721268,-0.696496,-0.721882,...,-0.178115,-0.173815,-0.177536,-0.180543,-0.177889,-0.179410,-0.177139,-0.177351,0.002094,6
8,2402.256283,1.599528,3.893632,0.029679,dart,250,"{'booster': 'dart', 'n_estimators': 250}",-0.693030,-0.667542,-0.696751,...,-0.168372,-0.164750,-0.167782,-0.169838,-0.166401,-0.170032,-0.168284,-0.167540,0.002249,3
9,3366.690917,1.824185,4.242734,0.338539,dart,300,"{'booster': 'dart', 'n_estimators': 300}",-0.676538,-0.652145,-0.683637,...,-0.161682,-0.156509,-0.158714,-0.162931,-0.157452,-0.158893,-0.159840,-0.159277,0.002054,2


Best parameters {'booster': 'dart', 'n_estimators': 300}
Best score -0.6922802578352828


The results are the same with gbtree and dart, but dart takes a lot longer to train. We will choose gbtree.

More estimators improves the results, we will test with more estimators to try and find the limit at which adding more estimators does not improve the results much

In [30]:
scoring = ["neg_root_mean_squared_error", "neg_mean_absolute_error"] # RMSE and MAE

cv_params = {'n_estimators': [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
            }

gscv_xgb_2 = GridSearchCV(XGBRegressor(),
                          param_grid = cv_params,
                          scoring = scoring,
                          cv = 10,
                          verbose = 2,
                          n_jobs = 10,
                          refit="neg_root_mean_squared_error"
)

gscv_xgb_2.fit(X_train,y_train)
df_2 = pd.DataFrame.from_dict(gscv_xgb_2.cv_results_)
pd.set_option('display.max_columns', None)
display(df_2)
print(f"Best parameters {gscv_xgb_2.best_params_}")
print(f"Best score {gscv_xgb_2.best_score_}")

Fitting 10 folds for each of 10 candidates, totalling 100 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,split3_test_neg_root_mean_squared_error,split4_test_neg_root_mean_squared_error,split5_test_neg_root_mean_squared_error,split6_test_neg_root_mean_squared_error,split7_test_neg_root_mean_squared_error,split8_test_neg_root_mean_squared_error,split9_test_neg_root_mean_squared_error,mean_test_neg_root_mean_squared_error,std_test_neg_root_mean_squared_error,rank_test_neg_root_mean_squared_error,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,split3_test_neg_mean_absolute_error,split4_test_neg_mean_absolute_error,split5_test_neg_mean_absolute_error,split6_test_neg_mean_absolute_error,split7_test_neg_mean_absolute_error,split8_test_neg_mean_absolute_error,split9_test_neg_mean_absolute_error,mean_test_neg_mean_absolute_error,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error
0,23.188663,0.054305,0.309838,0.005001,100,{'n_estimators': 100},-0.862622,-0.815595,-0.833992,-0.922820,-0.806739,-0.840190,-0.906333,-0.885421,-0.863118,-0.859691,-0.859652,0.035542,10,-0.208420,-0.203369,-0.210545,-0.210185,-0.206682,-0.205544,-0.212830,-0.211501,-0.215420,-0.205422,-0.208992,0.003579,10
1,41.873128,0.081130,0.587952,0.020353,200,{'n_estimators': 200},-0.721268,-0.696496,-0.721882,-0.773427,-0.690349,-0.740011,-0.776199,-0.735422,-0.740602,-0.746947,-0.734261,0.026842,9,-0.176690,-0.173655,-0.178720,-0.178115,-0.173815,-0.177536,-0.180543,-0.177889,-0.179410,-0.177139,-0.177351,0.002094,9
2,61.729190,0.149665,0.899116,0.037464,300,{'n_estimators': 300},-0.676538,-0.652146,-0.683637,-0.732628,-0.655791,-0.701772,-0.730894,-0.683911,-0.694332,-0.711155,-0.692280,0.026276,8,-0.159807,-0.156271,-0.160673,-0.161682,-0.156509,-0.158714,-0.162931,-0.157452,-0.158893,-0.159840,-0.159277,0.002054,8
3,81.396047,0.335990,1.383533,0.231670,400,{'n_estimators': 400},-0.650200,-0.631599,-0.663830,-0.703355,-0.637217,-0.682803,-0.709816,-0.655156,-0.672806,-0.689042,-0.669583,0.025358,7,-0.145691,-0.141360,-0.149428,-0.150069,-0.145260,-0.147627,-0.151945,-0.146064,-0.146950,-0.148499,-0.147289,0.002811,7
4,104.162949,0.490535,1.824301,0.408042,500,{'n_estimators': 500},-0.639257,-0.620956,-0.652298,-0.691265,-0.627228,-0.669581,-0.697537,-0.637327,-0.656773,-0.677557,-0.656978,0.025115,6,-0.140537,-0.134796,-0.143062,-0.141697,-0.138561,-0.140390,-0.144713,-0.137443,-0.138558,-0.140161,-0.139992,0.002696,6
5,124.983232,0.643087,2.361561,0.498340,600,{'n_estimators': 600},-0.633533,-0.612262,-0.640113,-0.685095,-0.622929,-0.660232,-0.689994,-0.631423,-0.652140,-0.670570,-0.649829,0.024969,5,-0.134402,-0.128508,-0.135870,-0.134720,-0.133726,-0.133314,-0.138347,-0.132661,-0.132750,-0.133764,-0.133806,0.002384,5
6,149.172118,0.717782,2.568778,0.636161,700,{'n_estimators': 700},-0.625569,-0.607863,-0.635133,-0.678575,-0.621181,-0.656236,-0.683569,-0.623403,-0.647427,-0.666657,-0.644561,0.024702,4,-0.127931,-0.124243,-0.129584,-0.129468,-0.128005,-0.128883,-0.132949,-0.127380,-0.128463,-0.128889,-0.128580,0.002052,4
7,158.095074,0.829726,3.024416,0.688507,800,{'n_estimators': 800},-0.619412,-0.603376,-0.631713,-0.675529,-0.619744,-0.652375,-0.680038,-0.620967,-0.643814,-0.665387,-0.641236,0.025037,3,-0.123464,-0.119408,-0.125406,-0.125477,-0.124861,-0.124767,-0.129388,-0.124073,-0.124972,-0.125587,-0.124740,0.002321,3
8,183.922738,0.963922,3.386511,0.884478,900,{'n_estimators': 900},-0.615941,-0.598248,-0.629587,-0.672776,-0.617748,-0.650625,-0.676012,-0.623025,-0.640578,-0.663642,-0.638818,0.024986,2,-0.119969,-0.116792,-0.122607,-0.122258,-0.121810,-0.121831,-0.125797,-0.121346,-0.122070,-0.122294,-0.121677,0.002136,2
9,205.257477,1.193017,3.881280,1.014430,1000,{'n_estimators': 1000},-0.614386,-0.597441,-0.626765,-0.671549,-0.616147,-0.648618,-0.674653,-0.620786,-0.639574,-0.662538,-0.

Best parameters {'n_estimators': 1000}
Best score -0.6372457587758518


600 estimators seem to be the place where adding more estimators does not improve the results a lot which is why we will use this value.

#### Tuning the max_depth and min_child_weight parameters

In [31]:
scoring = ["neg_root_mean_squared_error", "neg_mean_absolute_error"] # RMSE and MAE

cv_params = {'max_depth': [10, 30, 50],
             'min_child_weight': [3, 6, 9]
            }

gscv_xgb_2 = GridSearchCV(XGBRegressor(n_estimators=600
                                       ),
                          param_grid = cv_params,
                          scoring = scoring,
                          cv = 10,
                          verbose = 2,
                          n_jobs = 10,
                          refit="neg_root_mean_squared_error"
)

gscv_xgb_2.fit(X_train,y_train)
df_2 = pd.DataFrame.from_dict(gscv_xgb_2.cv_results_)
pd.set_option('display.max_columns', None)
display(df_2)
print(f"Best parameters {gscv_xgb_2.best_params_}")
print(f"Best score {gscv_xgb_2.best_score_}")

Fitting 10 folds for each of 9 candidates, totalling 90 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_min_child_weight,params,split0_test_neg_root_mean_squared_error,split1_test_neg_root_mean_squared_error,split2_test_neg_root_mean_squared_error,split3_test_neg_root_mean_squared_error,split4_test_neg_root_mean_squared_error,split5_test_neg_root_mean_squared_error,split6_test_neg_root_mean_squared_error,split7_test_neg_root_mean_squared_error,split8_test_neg_root_mean_squared_error,split9_test_neg_root_mean_squared_error,mean_test_neg_root_mean_squared_error,std_test_neg_root_mean_squared_error,rank_test_neg_root_mean_squared_error,split0_test_neg_mean_absolute_error,split1_test_neg_mean_absolute_error,split2_test_neg_mean_absolute_error,split3_test_neg_mean_absolute_error,split4_test_neg_mean_absolute_error,split5_test_neg_mean_absolute_error,split6_test_neg_mean_absolute_error,split7_test_neg_mean_absolute_error,split8_test_neg_mean_absolute_error,split9_test_neg_mean_absolute_error,mean_test_neg_mean_absolute_error,std_test_neg_mean_absolute_error,rank_test_neg_mean_absolute_error
0,216.996130,0.667281,7.282642,0.210698,10,3,"{'max_depth': 10, 'min_child_weight': 3}",-0.634528,-0.595696,-0.637307,-0.670742,-0.631100,-0.640625,-0.679338,-0.646013,-0.654347,-0.662486,-0.645218,0.022467,3,-0.090922,-0.087347,-0.093305,-0.093200,-0.092961,-0.091628,-0.093329,-0.093083,-0.094030,-0.093421,-0.092322,0.001872,7
1,201.464543,0.260862,6.658354,0.262661,10,6,"{'max_depth': 10, 'min_child_weight': 6}",-0.622431,-0.593026,-0.635545,-0.667754,-0.606278,-0.627444,-0.673003,-0.631883,-0.637604,-0.655768,-0.635073,0.024014,2,-0.092993,-0.090635,-0.095627,-0.095805,-0.094347,-0.093698,-0.096905,-0.094916,-0.096746,-0.095710,-0.094738,0.001810,8
2,203.622309,0.382889,6.067565,0.259780,10,9,"{'max_depth': 10, 'min_child_weight': 9}",-0.606851,-0.580455,-0.615109,-0.662099,-0.593661,-0.624549,-0.671083,-0.619358,-0.629215,-0.642587,-0.624497,0.026926,1,-0.094203,-0.091317,-0.096500,-0.098620,-0.095554,-0.095683,-0.098610,-0.096922,-0.097954,-0.097792,-0.096315,0.002151,9
3,353.765994,7.179277,20.134031,2.577491,30,3,"{'max_depth': 30, 'min_child_weight': 3}",-0.670135,-0.627377,-0.659862,-0.701313,-0.665351,-0.684749,-0.712138,-0.665009,-0.659563,-0.701340,-0.674684,0.024107,8,-0.084372,-0.080872,-0.086112,-0.086548,-0.086458,-0.085749,-0.086865,-0.085664,-0.086724,-0.087251,-0.085662,0.001770,6
4,453.526178,6.614705,37.690233,4.308708,30,6,"{'max_depth': 30, 'min_child_weight': 6}",-0.660757,-0.624293,-0.659833,-0.692973,-0.633186,-0.667832,-0.695514,-0.649037,-0.658476,-0.667054,-0.660896,0.021374,6,-0.083676,-0.081293,-0.086346,-0.086655,-0.085074,-0.085242,-0.086272,-0.084971,-0.087149,-0.086149,-0.085283,0.001639,4
5,430.583238,5.041421,40.415308,4.444990,30,9,"{'max_depth': 30, 'min_child_weight': 9}",-0.648864,-0.614550,-0.644799,-0.691672,-0.641208,-0.644558,-0.681922,-0.644334,-0.646199,-0.674072,-0.653218,0.021610,5,-0.083471,-0.080895,-0.085753,-0.086411,-0.085308,-0.084141,-0.085761,-0.084794,-0.086464,-0.087104,-0.085010,0.001726,2
6,373.085914,4.543448,27.919495,1.851947,50,3,"{'max_depth': 50, 'min_child_weight': 3}",-0.673901,-0.625940,-0.660902,-0.701331,-0.664841,-0.685445,-0.714674,-0.666376,-0.659631,-0.704262,-0.675730,0.024996,9,-0.084407,-0.080633,-0.086152,-0.086564,-0.086359,-0.085645,-0.086755,-0.085768,-0.086693,-0.087416,-0.085639,0.001837,5
7,508.846024,5.407818,45.821680,6.532356,50,6,"{'max_depth': 50, 'min_child_weight': 6}",-0.662593,-0.620849,-0.662466,-0.694651,-0.635007,-0.669380,-0.693050,-0.647182,-0.658117,-0.668568,-0.661186,0.021860,7,-0.083623,-0.081006,-0.086667,-0.086418,-0.085034,-0.085149,-0.086112,-0.084633,-0.087125,-0.086212,-0.085198,0.001721,3
8,603.883579,8.063307,44.362552,10.541659,50,9,"{'max_depth': 50, 'min_child_weight': 9}",-0.645829,-0.610272,-0.642087,-0.692493,-0.644174,-0.646492,-0.682432,-0.640201,-0.648872,-0.677095,-0.652995,0.023030,4,-0.083375,-0.080499,-0.085306,-0.086304,-0.085422,-0.084364,-0.08

Best parameters {'max_depth': 10, 'min_child_weight': 9}
Best score -0.6244966662720265


In [ ]:
scoring = ["neg_root_mean_squared_error", "neg_mean_absolute_error"] # RMSE and MAE

cv_params = {'max_depth': [5, 10, 15],
             'min_child_weight': [7, 9, 11]
            }

gscv_xgb_2 = GridSearchCV(XGBRegressor(n_estimators=600
                                       ),
                          param_grid = cv_params,
                          scoring = scoring,
                          cv = 10,
                          verbose = 2,
                          n_jobs = 10,
                          refit="neg_root_mean_squared_error"
)

gscv_xgb_2.fit(X_train,y_train)
df_2 = pd.DataFrame.from_dict(gscv_xgb_2.cv_results_)
pd.set_option('display.max_columns', None)
display(df_2)
print(f"Best parameters {gscv_xgb_2.best_params_}")
print(f"Best score {gscv_xgb_2.best_score_}")

In [ ]:
def XGB_objective(trial, X_train, y_train, X_test, y_test, output_parameters):
    # Hyperparameters which will be tuned
    booster = trial.suggest_categorical("booster", ["gbtree", "dart"])
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 3, 50)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    gamma = trial.suggest_int("gamma", 0, 10)
    learning_rate = trial.suggest_float("learning_rate", 0.1, 0.4)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    n_jobs = 10

    mdl = XGBRegressor(
        booster = booster,
        n_estimators = n_estimators,
        max_depth = max_depth,
        min_child_weight = min_child_weight,
        gamma = gamma,
        learning_rate = learning_rate,
        subsample = subsample,
        n_jobs = n_jobs
    )
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test) # shape of preds = [[mass_1, radius_1], [mass_2, radius_2], ...]
    # truth, preds = Model_trainer.Kfold_pipeline(XGBRegressor, X_train_data=X_train, y_train_data=y_train, n_splits=10, 
    #                                             max_depth = max_depth,
    #                                             min_child_weight = min_child_weight,
    #                                             gamma = gamma,
    #                                             learning_rate = learning_rate,
    #                                             subsample = subsample,
    #                                             n_jobs = n_jobs)
    metrics_dict = dict()
    for i, output_param in enumerate(output_parameters):
        metrics_dict[output_param] = dict()
        metrics_dict[output_param]["RMSE"] = root_mean_squared_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAE"] = mean_absolute_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAX_ER"] = max_error(y_test[:, i], preds[:, i])


    return metrics_dict["mass"]["RMSE"], metrics_dict["mass"]["MAE"], metrics_dict["mass"]["MAX_ER"]

In [10]:
study_XGB = optuna.create_study(directions=["minimize", "minimize", "minimize"]) # we want to minimize the RMSE, MAE and MAX_ER
study_XGB.optimize(lambda trial: XGB_objective(trial, X_train, y_train, X_test, y_test, output_parameters), n_trials=100)

[I 2025-12-14 12:21:14,303] A new study created in memory with name: no-name-11f341a6-754c-4986-a6cc-7dabe2b60931
[I 2025-12-14 12:29:16,266] Trial 0 finished with values: [1.2224247642157515, 0.1571416991172521, 69.67560290270555] and parameters: {'booster': 'dart', 'n_estimators': 163, 'max_depth': 28, 'min_child_weight': 10, 'gamma': 0, 'learning_rate': 0.23085215736278564, 'subsample': 0.8256730845862008}.
[I 2025-12-14 12:29:19,010] Trial 1 finished with values: [1.284814627980934, 0.216067937860846, 81.08542345934617] and parameters: {'booster': 'gbtree', 'n_estimators': 104, 'max_depth': 35, 'min_child_weight': 4, 'gamma': 4, 'learning_rate': 0.3121100779260943, 'subsample': 0.7056502750801965}.
[W 2025-12-14 12:29:51,146] Trial 2 failed with parameters: {'booster': 'dart', 'n_estimators': 188, 'max_depth': 44, 'min_child_weight': 10, 'gamma': 3, 'learning_rate': 0.341730349340317, 'subsample': 0.6829671400052018} because of the following error: KeyboardInterrupt().
Traceback (m

KeyboardInterrupt: 

In [9]:
for trial in study_XGB.best_trials:
    print(f"[RMSE, MAE, MAX_ER] : {trial.values} \n parameters : {trial.params}")


[RMSE, MAE, MAX_ER] : [1.2316886554136264, 0.22190515384497075, 62.295323361689924] 
 parameters : {'booster': 'dart', 'n_estimators': 172, 'max_depth': 22, 'min_child_weight': 9, 'gamma': 6, 'learning_rate': 0.24524589096827268, 'subsample': 0.5658450568700828}
[RMSE, MAE, MAX_ER] : [1.1838174487353543, 0.1584404693204412, 64.04990099840867] 
 parameters : {'booster': 'gbtree', 'n_estimators': 104, 'max_depth': 40, 'min_child_weight': 8, 'gamma': 0, 'learning_rate': 0.19241785132798528, 'subsample': 0.5078459944304237}
[RMSE, MAE, MAX_ER] : [1.2335819738583709, 0.2185464409930592, 57.57323560324306] 
 parameters : {'booster': 'gbtree', 'n_estimators': 163, 'max_depth': 20, 'min_child_weight': 7, 'gamma': 9, 'learning_rate': 0.2655980790371276, 'subsample': 0.9604583427104191}
[RMSE, MAE, MAX_ER] : [1.2478721521180527, 0.21502093161805308, 61.73708056383836] 
 parameters : {'booster': 'dart', 'n_estimators': 239, 'max_depth': 41, 'min_child_weight': 5, 'gamma': 9, 'learning_rate': 0.15

In [8]:
# [RMSE, MAE, MAX_ER] : [1.2316886554136264, 0.22190515384497075, 62.295323361689924] 
#  parameters : {'booster': 'dart', 'n_estimators': 172, 'max_depth': 22, 'min_child_weight': 9, 'gamma': 6, 'learning_rate': 0.24524589096827268, 'subsample': 0.5658450568700828}
best_params_XGB = {'booster': 'dart', 'n_estimators': 172, 'max_depth': 22, 'min_child_weight': 9, 'gamma': 6, 'learning_rate': 0.24524589096827268, 'subsample': 0.5658450568700828}

print(path_to_results)
xgb_evaluator = Model_evaluator("XGBoost", path=path_to_results, physical_model=physical_model, output_parameters=["mass", "radius"])
xgb_evaluator.evaluate_Kfold_results(XGBRegressor, X_train, y_train, path_to_predictions, tag, random_state=12, override=True, use_preds=False, **best_params_XGB, n_jobs=10)

../../../../../../../results/model_A/fine_tuned_models/
Error: path does not exist.


SystemExit: 1

c:\Users\antoi\Code\unif\MA2\thesis\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Random forest

In [ ]:
def RF_objective(trial, X_train, y_train, X_test, y_test):
    # Hyperparameters which will be tuned
    criterion = trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "poisson", "absolute_error"])
    # n_estimators = trial.suggest_int("n_estimators", 75, 125)
    max_depth = trial.suggest_int("max_depth", 3, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)

    mdl = RandomForestRegressor(
        criterion = criterion,
        max_depth = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf = min_samples_leaf,
        n_jobs = 10
    )
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test)
    # truth, preds = Model_trainer.Kfold_pipeline(RandomForestRegressor, X_train_data=X_train, y_train_data=y_train, n_splits=10, 
    #                                             criterion = criterion,
    #                                             # n_estimators = n_estimators,
    #                                             max_depth = max_depth,
    #                                             min_samples_split = min_samples_split,
    #                                             min_samples_leaf = min_samples_leaf,
    #                                             n_jobs = 10
    #                                             )
    metrics_dict = dict()
    for i, output_param in enumerate(output_parameters):
        metrics_dict[output_param] = dict()
        metrics_dict[output_param]["RMSE"] = root_mean_squared_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAE"] = mean_absolute_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAX_ER"] = max_error(y_test[:, i], preds[:, i])

    return metrics_dict["mass"]["RMSE"], metrics_dict["mass"]["MAE"], metrics_dict["mass"]["MAX_ER"]

In [ ]:
study_RF = optuna.create_study(directions=["minimize", "minimize", "minimize"]) # we want to minimize the RMSE, MAE and MAX_ER
study_RF.optimize(lambda trial: RF_objective(trial, X_train, y_train, X_test, y_test, output_parameters), n_trials=100)

In [ ]:
for trial in study_RF.best_trials:
    print(f"[RMSE, MAE, MAX_ER] : {trial.values} \n parameters : {trial.params}")

In [ ]:

best_params_RF = 

rf_evaluator = Model_evaluator("radnom_forest", path=path_to_results, physical_model=physical_model, output_parameters=["mass", "radius"])
rf_evaluator.evaluate_Kfold_results(RandomForestRegressor, X_train, y_train, path_to_predictions, tag, random_state=12, override=True, use_preds=False, **best_params_RF, n_jobs=10)